# Download Full Economist Page Images from Deduplicated Face Data

This notebook downloads native-size Economist page images for source scan entries in the deduplicated face-detection dataset. It is intended to be run from `code/scripts`, matching the repository's notebook execution convention.

Default input:

`../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv`

Default output directory:

`../../data/images/full_pages_1940_2007/`

The selection is restricted to issues from 1940 through 2007. A source scan entry can encode one page or a two-page spread such as `1996-0413-0045,0046`; when such an entry is selected, every encoded page is considered for download. Existing downloaded pages are detected before sampling and are skipped unless `OVERWRITE = True`.


## Parameters

`RUN_MODE = "random"` samples source scan entries after excluding entries whose pages are already downloaded. `RUN_MODE = "sequential"` selects every eligible source scan in chronological order. In both modes, multi-page scan entries are expanded before download so the output files are individual page images named `yyyy-mmdd-pppp.<detected filetype>`.

Downloads are serial by default. Set `PARALLEL_DOWNLOADS = True` to use a small thread pool. Each worker owns its own authenticated `requests.Session`, and the main thread collects results before writing manifests. Keep `MAX_WORKERS` low at first because the Nationallizenzen/IIIF service may throttle or redirect aggressively under load.

The remote image request uses the IIIF native-size form `/full/full/0/default.JPG`, resolved from the local METS `DEFAULT` image URL. If a logged-in session uses a different archive host than the METS files, set `IMAGE_BASE_URL_OVERRIDE` to that authenticated host, for example `"https://...zugang.nationallizenzen.de"`.


In [ ]:
from pathlib import Path


RUN_MODE = "sequential"  # "random" or "sequential"
START_YEAR = 1940
END_YEAR = 2007
RANDOM_SAMPLE_SOURCE_SCANS = 2000
RANDOM_SEED = 42

DOWNLOAD_IMAGES = True
TEST_DOWNLOAD_PAGE_LIMIT = None  # Set to an integer for a bounded smoke test.
OVERWRITE = False

PARALLEL_DOWNLOADS = False
MAX_WORKERS = 1
CHECKPOINT_EVERY_RECORDS = 100  # Set to None to disable periodic checkpoint manifests.

# Paths are relative to this notebook's directory: code/scripts.
DEDUPLICATED_CSV = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv")
METADATA_ROOT = Path("../../data/metadata")
AUTH_COOKIE_PATH = Path("../auth/nationallizenzen_cookie.json")
OUTPUT_DIR = Path("../../data/images/full_pages_1940_2007")

FULL_SIZE_SEGMENT = "full"  # "full" and "max" both request native size on this IIIF service.
IMAGE_BASE_URL_OVERRIDE = None

REQUEST_TIMEOUT = 90
REQUEST_SLEEP_SECONDS = 0.1
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 2.0
AUTH_FAILURE_LIMIT = 1  # Stop queued work after this many authentication-like responses.
MIN_VALID_IMAGE_BYTES = 1_000
RETRY_HTTP_STATUSES = {429, 500, 502, 503, 504}


In [ ]:
import json
import random
import re
import time
import xml.etree.ElementTree as ET
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from threading import Event, Lock, local
from urllib.parse import urlparse, urlunparse

import pandas as pd
import requests


pd.options.display.max_rows = 120
pd.options.display.max_columns = 80
pd.options.display.max_colwidth = 180

PAGE_ID_RE = re.compile(r"^(?P<issue_id>\d{4}-\d{4})-(?P<page_number>\d{4})$")
FILENAME_RE = re.compile(r"^(?P<issue_id>\d{4}-\d{4})-(?P<source_pages>\d{4}(?:,\d{4})*)")
IMAGE_FILENAME_RE = re.compile(r"^(?P<page_id>\d{4}-\d{4}-\d{4})\.[^.]+$")
IIIF_DEFAULT_SIZE_RE = re.compile(r"/full/[^/]+/0/default\.[A-Za-z]+$")
XLINK_HREF = "{http://www.w3.org/1999/xlink}href"

AUTH_FAILURE_MARKERS = (
    "Nationallizenzen Web Anmeldedienst",
    "login.nationallizenzen.de",
    "SAML2/Redirect/SSO",
    "Shibboleth",
)

SELECTED_SCANS_CSV = OUTPUT_DIR / "selected_source_scans.csv"
SELECTED_SCANS_JSON = OUTPUT_DIR / "selected_source_scans.json"
SELECTED_PAGE_IDS_JSON = OUTPUT_DIR / "selected_page_ids.json"
PLANNED_PAGES_CSV = OUTPUT_DIR / "planned_pages.csv"
MANIFEST_CSV = OUTPUT_DIR / "download_manifest.csv"
MANIFEST_JSON = OUTPUT_DIR / "download_manifest.json"
CHECKPOINT_JSON = OUTPUT_DIR / "download_manifest_checkpoint.json"
ERROR_CSV = OUTPUT_DIR / "download_errors.csv"
ERROR_JSON = OUTPUT_DIR / "download_errors.json"

run_mode = str(RUN_MODE).lower().strip()
full_size_segment = str(FULL_SIZE_SEGMENT).strip().lower()
max_workers = int(MAX_WORKERS)

assert run_mode in {"random", "sequential"}, RUN_MODE
assert START_YEAR <= END_YEAR
assert RANDOM_SAMPLE_SOURCE_SCANS is None or RANDOM_SAMPLE_SOURCE_SCANS > 0
assert TEST_DOWNLOAD_PAGE_LIMIT is None or TEST_DOWNLOAD_PAGE_LIMIT > 0
assert full_size_segment in {"full", "max"}, FULL_SIZE_SEGMENT
assert max_workers >= 1
assert CHECKPOINT_EVERY_RECORDS is None or CHECKPOINT_EVERY_RECORDS > 0
assert REQUEST_TIMEOUT > 0
assert REQUEST_SLEEP_SECONDS >= 0
assert MAX_RETRIES >= 1
assert RETRY_BACKOFF_SECONDS >= 0
assert AUTH_FAILURE_LIMIT >= 1
assert MIN_VALID_IMAGE_BYTES >= 0
assert DEDUPLICATED_CSV.exists(), f"Missing input CSV: {DEDUPLICATED_CSV.resolve()}"
assert METADATA_ROOT.exists(), f"Missing METS metadata directory: {METADATA_ROOT.resolve()}"
assert AUTH_COOKIE_PATH.exists(), f"Missing auth cookie file: {AUTH_COOKIE_PATH.resolve()}"

if IMAGE_BASE_URL_OVERRIDE is not None:
    parsed_override = urlparse(str(IMAGE_BASE_URL_OVERRIDE))
    assert parsed_override.scheme and parsed_override.netloc, IMAGE_BASE_URL_OVERRIDE

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input CSV: {DEDUPLICATED_CSV.resolve()}")
print(f"METS root: {METADATA_ROOT.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Mode: {run_mode}")
print(f"Overwrite existing files: {OVERWRITE}")
print(f"Parallel downloads: {PARALLEL_DOWNLOADS} (MAX_WORKERS={max_workers})")


## Load and Validate Deduplicated Data

The deduplicated face CSV is used as the source of issue/page entries. The filename prefix identifies the issue and the source page or pages. The notebook collapses repeated detections to one row per source scan entry before sampling, while retaining detection counts and maximum confidence for the selection manifest.


In [ ]:
expected_columns = [
    "Filename",
    "Bounding Box relative X1",
    "Bounding Box relative Y1",
    "Bounding Box relative X2",
    "Bounding Box relative Y2",
    "Segmentation confidence score",
    "Size relative",
    "Age",
    "Gender",
]

faces = pd.read_csv(DEDUPLICATED_CSV, dtype={"Filename": "string"})
assert list(faces.columns) == expected_columns, {
    "expected": expected_columns,
    "actual": list(faces.columns),
}
assert len(faces) > 0, "The deduplicated CSV is empty."

filename_parts = faces["Filename"].str.extract(FILENAME_RE)
parse_failures = int(filename_parts["issue_id"].isna().sum())
assert parse_failures == 0, f"Could not parse {parse_failures:,} filenames."

faces = faces.assign(
    issue_id=filename_parts["issue_id"],
    source_pages=filename_parts["source_pages"],
    source_scan_id=(
        faces["Filename"]
        .str.replace(r"\.[^.]+$", "", regex=True)
        .str.split("_", n=1)
        .str[0]
    ),
)
faces["issue_date"] = pd.to_datetime(faces["issue_id"], format="%Y-%m%d")
faces["year"] = faces["issue_date"].dt.year
faces["Segmentation confidence score"] = pd.to_numeric(faces["Segmentation confidence score"], errors="coerce")
assert faces["Segmentation confidence score"].notna().all(), "Confidence scores must be numeric."

faces_in_range = faces.loc[faces["year"].between(START_YEAR, END_YEAR)].copy()
assert len(faces_in_range) > 0, f"No deduplicated rows found for {START_YEAR}-{END_YEAR}."

scan_consistency = faces_in_range.groupby("source_scan_id", sort=False).agg(
    issue_ids=("issue_id", "nunique"),
    source_page_values=("source_pages", "nunique"),
)
inconsistent_scans = scan_consistency.loc[
    (scan_consistency["issue_ids"] != 1) | (scan_consistency["source_page_values"] != 1)
]
assert inconsistent_scans.empty, inconsistent_scans.head().to_dict("index")

source_scans = (
    faces_in_range.groupby("source_scan_id", as_index=False, sort=False)
    .agg(
        issue_id=("issue_id", "first"),
        source_pages=("source_pages", "first"),
        issue_date=("issue_date", "first"),
        year=("year", "first"),
        first_filename=("Filename", "first"),
        detection_rows=("Filename", "size"),
        max_confidence=("Segmentation confidence score", "max"),
    )
    .sort_values(["issue_date", "source_scan_id"], kind="mergesort")
    .reset_index(drop=True)
)

def page_ids_for_scan(row: pd.Series) -> list[str]:
    page_numbers = [page.strip() for page in str(row["source_pages"]).split(",") if page.strip()]
    assert page_numbers, row.to_dict()
    assert all(re.fullmatch(r"\d{4}", page_number) for page_number in page_numbers), page_numbers
    return [f"{row['issue_id']}-{page_number}" for page_number in page_numbers]

source_scans["page_ids"] = source_scans.apply(page_ids_for_scan, axis=1)
source_scans["page_count"] = source_scans["page_ids"].map(len)

assert source_scans["source_scan_id"].is_unique
assert source_scans["page_count"].ge(1).all()
assert source_scans["page_ids"].explode().map(lambda page_id: PAGE_ID_RE.fullmatch(page_id) is not None).all()

summary = pd.Series(
    {
        "deduplicated_face_rows": len(faces),
        "rows_in_year_range": len(faces_in_range),
        "unique_source_scans_in_range": len(source_scans),
        "single_page_source_scans": int((source_scans["page_count"] == 1).sum()),
        "multi_page_source_scans": int((source_scans["page_count"] > 1).sum()),
        "unique_pages_after_expansion": source_scans["page_ids"].explode().nunique(),
        "year_min": int(source_scans["year"].min()),
        "year_max": int(source_scans["year"].max()),
    },
    name="value",
)
display(summary.to_frame())
source_scans.head(10)


## Check Existing Downloads and Select Pages

Existing images are detected by filename stem and magic bytes, not by extension alone. In random mode, the notebook samples only source scan entries with at least one missing page unless `OVERWRITE = True`. In sequential mode, all eligible entries are selected chronologically.


In [ ]:
def detect_image_extension(data: bytes) -> str:
    header = bytes(data[:32])
    if header.startswith(b"\xff\xd8\xff"):
        return ".jpg"
    if header.startswith(b"\x89PNG\r\n\x1a\n"):
        return ".png"
    if header.startswith(b"GIF87a") or header.startswith(b"GIF89a"):
        return ".gif"
    if header.startswith(b"RIFF") and header[8:12] == b"WEBP":
        return ".webp"
    if header.startswith(b"II*\x00") or header.startswith(b"MM\x00*"):
        return ".tif"
    raise ValueError(f"Unrecognized image signature: {header[:16]!r}")

def image_magic_extension(path: Path) -> str | None:
    try:
        with path.open("rb") as file:
            return detect_image_extension(file.read(32))
    except Exception:
        return None

def existing_image_paths_for_page_id(page_id: str) -> list[Path]:
    return sorted(path for path in OUTPUT_DIR.glob(f"{page_id}.*") if path.is_file())

def valid_existing_image_paths_for_page_id(page_id: str) -> list[Path]:
    valid_paths = []
    for path in existing_image_paths_for_page_id(page_id):
        if path.stat().st_size < MIN_VALID_IMAGE_BYTES:
            continue
        if image_magic_extension(path) is None:
            continue
        valid_paths.append(path)
    return valid_paths

def downloaded_page_ids(output_dir: Path) -> set[str]:
    page_ids = set()
    for path in output_dir.iterdir() if output_dir.exists() else []:
        match = IMAGE_FILENAME_RE.fullmatch(path.name)
        if match is None or not path.is_file():
            continue
        page_id = match.group("page_id")
        if path.stat().st_size >= MIN_VALID_IMAGE_BYTES and image_magic_extension(path) is not None:
            page_ids.add(page_id)
    return page_ids

existing_page_ids = downloaded_page_ids(OUTPUT_DIR)

source_scans = source_scans.copy()
source_scans["missing_page_ids"] = source_scans["page_ids"].map(
    lambda page_ids: [page_id for page_id in page_ids if page_id not in existing_page_ids]
)
source_scans["downloaded_page_count"] = source_scans["page_ids"].map(
    lambda page_ids: sum(page_id in existing_page_ids for page_id in page_ids)
)

if OVERWRITE:
    eligible_scans = source_scans.copy()
else:
    eligible_scans = source_scans.loc[source_scans["missing_page_ids"].map(bool)].copy()

if run_mode == "random":
    if RANDOM_SAMPLE_SOURCE_SCANS is None:
        sample_count = len(eligible_scans)
    else:
        sample_count = min(int(RANDOM_SAMPLE_SOURCE_SCANS), len(eligible_scans))
    selected_scans = (
        eligible_scans.sample(n=sample_count, random_state=RANDOM_SEED)
        if sample_count
        else eligible_scans.head(0)
    )
    selected_scans = selected_scans.sort_values(["issue_date", "source_scan_id"], kind="mergesort")
elif run_mode == "sequential":
    selected_scans = eligible_scans.sort_values(["issue_date", "source_scan_id"], kind="mergesort")
else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

selected_scans = selected_scans.reset_index(drop=True)

target_page_rows = []
for _, scan in selected_scans.iterrows():
    for page_id in scan["page_ids"]:
        existing_paths = valid_existing_image_paths_for_page_id(page_id)
        target_page_rows.append(
            {
                "page_id": page_id,
                "issue_id": scan["issue_id"],
                "issue_date": scan["issue_date"].date().isoformat(),
                "year": int(scan["year"]),
                "source_scan_id": scan["source_scan_id"],
                "source_pages": scan["source_pages"],
                "source_page_count": int(scan["page_count"]),
                "detection_rows": int(scan["detection_rows"]),
                "max_confidence": float(scan["max_confidence"]),
                "already_downloaded": bool(existing_paths),
                "existing_paths": [str(path) for path in existing_paths],
            }
        )

target_pages = pd.DataFrame(
    target_page_rows,
    columns=[
        "page_id",
        "issue_id",
        "issue_date",
        "year",
        "source_scan_id",
        "source_pages",
        "source_page_count",
        "detection_rows",
        "max_confidence",
        "already_downloaded",
        "existing_paths",
    ],
)

if not target_pages.empty:
    source_scan_ids_by_page = target_pages.groupby("page_id")["source_scan_id"].agg(
        lambda values: ";".join(sorted(set(map(str, values))))
    )
    target_pages = (
        target_pages.sort_values(["issue_date", "page_id", "source_scan_id"], kind="mergesort")
        .drop_duplicates(subset="page_id", keep="first")
        .reset_index(drop=True)
    )
    target_pages["source_scan_ids_for_page"] = target_pages["page_id"].map(source_scan_ids_by_page)
else:
    target_pages["source_scan_ids_for_page"] = pd.Series(dtype="string")

selected_scans_for_export = selected_scans.copy()
if not selected_scans_for_export.empty:
    selected_scans_for_export["issue_date"] = selected_scans_for_export["issue_date"].dt.date.astype(str)
    selected_scans_for_export["page_ids"] = selected_scans_for_export["page_ids"].map(";".join)
    selected_scans_for_export["missing_page_ids"] = selected_scans_for_export["missing_page_ids"].map(";".join)
selected_scans_for_export.to_csv(SELECTED_SCANS_CSV, index=False)
SELECTED_SCANS_JSON.write_text(
    json.dumps(selected_scans_for_export.to_dict("records"), indent=2, default=str) + "\n",
    encoding="utf-8",
)
SELECTED_PAGE_IDS_JSON.write_text(
    json.dumps(target_pages["page_id"].tolist(), indent=2) + "\n",
    encoding="utf-8",
)
target_pages.to_csv(PLANNED_PAGES_CSV, index=False)

selection_summary = pd.Series(
    {
        "existing_downloaded_pages_detected": len(existing_page_ids),
        "eligible_source_scans": len(eligible_scans),
        "selected_source_scans": len(selected_scans),
        "planned_unique_pages": len(target_pages),
        "planned_pages_already_downloaded": int(target_pages["already_downloaded"].sum()) if not target_pages.empty else 0,
        "selected_scans_csv": str(SELECTED_SCANS_CSV),
        "planned_pages_csv": str(PLANNED_PAGES_CSV),
    },
    name="value",
)
display(selection_summary.to_frame())
target_pages.head(20)


## Resolve Native-Size IIIF URLs from METS

Each page ID maps to `../../data/metadata/<year>/ECON-yyyy-mmdd.mets.xml`. The notebook reads the `DEFAULT` file group, finds the page image URL, and replaces the advertised derivative size with `/full/full/0/default.JPG` (or `/full/max/0/default.JPG` if configured).


In [ ]:
def mets_path_for_page_id(page_id: str) -> Path:
    match = PAGE_ID_RE.fullmatch(page_id)
    assert match is not None, page_id
    issue_id = match.group("issue_id")
    return METADATA_ROOT / issue_id[:4] / f"ECON-{issue_id}.mets.xml"

def full_iiif_url_from_href(href: str) -> str:
    parsed = urlparse(href)
    full_path = IIIF_DEFAULT_SIZE_RE.sub(f"/full/{full_size_segment}/0/default.JPG", parsed.path)
    if full_path == parsed.path:
        raise ValueError(f"Could not replace IIIF size segment in URL: {href}")

    if IMAGE_BASE_URL_OVERRIDE is not None:
        override = urlparse(str(IMAGE_BASE_URL_OVERRIDE))
        parsed = parsed._replace(scheme=override.scheme, netloc=override.netloc)

    return urlunparse(parsed._replace(path=full_path))

def default_href_from_mets(page_id: str) -> str:
    match = PAGE_ID_RE.fullmatch(page_id)
    assert match is not None, page_id
    issue_id = match.group("issue_id")
    page_number = match.group("page_number")
    mets_path = mets_path_for_page_id(page_id)
    assert mets_path.exists(), f"Missing METS file for {page_id}: {mets_path}"

    root = ET.parse(mets_path).getroot()
    target_fragment = f"ECON-{issue_id}-{page_number}"
    available_sizes = []

    for file_group in root.findall(".//{*}fileGrp"):
        group_size = str(file_group.attrib.get("USE", "")).upper()
        if group_size:
            available_sizes.append(group_size)
        if group_size != "DEFAULT":
            continue

        for location in file_group.findall(".//{*}FLocat"):
            href = location.attrib.get(XLINK_HREF)
            if href and target_fragment in href:
                return href

    raise ValueError(
        f"Could not find DEFAULT image URL for {page_id}. "
        f"Available METS fileGrp USE values: {sorted(set(available_sizes))}"
    )

resolved_records = []
for page in target_pages.to_dict("records"):
    default_href = default_href_from_mets(page["page_id"])
    resolved_records.append(
        {
            **page,
            "mets_path": str(mets_path_for_page_id(page["page_id"])),
            "default_image_url": default_href,
            "full_image_url": full_iiif_url_from_href(default_href),
            "output_stem": str(OUTPUT_DIR / page["page_id"]),
            "requested_iiif_size": full_size_segment,
        }
    )

resolved_pages = pd.DataFrame(resolved_records)
if not resolved_pages.empty:
    assert resolved_pages["page_id"].is_unique
    assert resolved_pages["full_image_url"].str.contains(f"/full/{full_size_segment}/0/default.JPG", regex=False).all()

print(f"Resolved {len(resolved_pages):,} native-size image URLs.")
resolved_pages.head(20)


## Authentication

The cookie file must contain `HANID` and `HHAUTHID`, matching the METS download workflow. The downloader treats redirects or small text responses containing Nationallizenzen login markers as authentication failures and stops after repeated consecutive auth-like failures.


In [ ]:
def load_auth_cookies(path: Path) -> dict[str, str]:
    required = {"HANID", "HHAUTHID"}
    cookies = json.loads(path.read_text(encoding="utf-8"))
    missing = sorted(required - set(cookies)) if isinstance(cookies, dict) else sorted(required)
    if missing:
        raise ValueError(f"Missing cookie fields in {path}: {missing}")
    if not all(isinstance(cookies[name], str) and cookies[name].strip() for name in required):
        raise ValueError(f"Cookie values in {path} must be non-empty strings")
    return {name: cookies[name].strip() for name in sorted(required)}

REQUEST_HEADERS = {
    "User-Agent": "master-thesis-full-page-download/1.0",
    "Accept": "image/avif,image/webp,image/png,image/jpeg,image/*;q=0.8,*/*;q=0.5",
    "Connection": "keep-alive",
}
AUTH_COOKIES = load_auth_cookies(AUTH_COOKIE_PATH)
_thread_state = local()

def make_session() -> requests.Session:
    session = requests.Session()
    session.headers.update(REQUEST_HEADERS)
    session.cookies.update(AUTH_COOKIES)
    return session

def session_for_current_thread() -> requests.Session:
    if not hasattr(_thread_state, "session"):
        _thread_state.session = make_session()
    return _thread_state.session

print(f"Loaded cookies from {AUTH_COOKIE_PATH}")


## Download Images

Downloads are serial unless `PARALLEL_DOWNLOADS = True`. In parallel mode, the notebook uses a bounded `ThreadPoolExecutor`, one authenticated session per worker thread, and a shared auth-stop flag. If authentication appears to fail, no new work is queued after the configured auth-failure limit is reached.

The retry loop is limited to transient HTTP statuses and request exceptions. Image type is detected from magic bytes before writing, so a server response with PNG bytes is saved as `.png` even if the URL ends in `.JPG`.


In [ ]:
class AuthenticationFailure(RuntimeError):
    pass

class AuthStopRequested(RuntimeError):
    pass


def response_looks_like_auth_failure(response: requests.Response) -> bool:
    content_type = response.headers.get("Content-Type", "")
    text_sample = ""
    if "text" in content_type.lower() or len(response.content) < 4096:
        text_sample = response.text[:5000]
    auth_text = response.url + "\n" + text_sample
    if response.status_code in {401, 403}:
        return True
    return any(marker in auth_text for marker in AUTH_FAILURE_MARKERS)


def fetch_with_retries(url: str, label: str, auth_stop_event: Event | None = None) -> tuple[requests.Response, int]:
    last_exception = None
    for attempt in range(1, MAX_RETRIES + 1):
        if auth_stop_event is not None and auth_stop_event.is_set():
            raise AuthStopRequested("Authentication stop requested before retrying.")
        try:
            response = session_for_current_thread().get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
            if response_looks_like_auth_failure(response):
                raise AuthenticationFailure(
                    f"Authentication failed while fetching {label}. Final URL: {response.url}. "
                    f"Refresh {AUTH_COOKIE_PATH} from a logged-in Nationallizenzen session."
                )
            if response.status_code in RETRY_HTTP_STATUSES and attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SECONDS * attempt)
                continue
            response.raise_for_status()
            return response, attempt
        except AuthenticationFailure:
            raise
        except requests.RequestException as exc:
            last_exception = exc
            if attempt >= MAX_RETRIES:
                raise
            time.sleep(RETRY_BACKOFF_SECONDS * attempt)
    raise RuntimeError(f"Request failed for {label}: {last_exception}")


def remove_stale_sibling_images(page_id: str, keep_path: Path) -> None:
    for path in existing_image_paths_for_page_id(page_id):
        if path != keep_path:
            path.unlink()


def write_detected_image(page_id: str, image_bytes: bytes) -> tuple[Path, str]:
    detected_extension = detect_image_extension(image_bytes)
    output_path = OUTPUT_DIR / f"{page_id}{detected_extension}"
    tmp_path = output_path.with_suffix(output_path.suffix + ".tmp")
    tmp_path.write_bytes(image_bytes)
    tmp_path.replace(output_path)
    if OVERWRITE:
        remove_stale_sibling_images(page_id, output_path)
    return output_path, detected_extension


def record_page_status(page: dict, status: str, **values) -> dict:
    record = {
        **page,
        "status": status,
        "output_path": None,
        "bytes": None,
        "detected_extension": None,
        "content_type": None,
        "final_url": None,
        "attempts": 0,
        "error": None,
    }
    record.update(values)
    return record


def download_page(page: dict, auth_stop_event: Event, auth_state: dict, auth_state_lock: Lock) -> dict:
    page_id = page["page_id"]
    if auth_stop_event.is_set():
        return record_page_status(page, "not_processed_auth_stop")

    try:
        existing_paths = valid_existing_image_paths_for_page_id(page_id)
        if existing_paths and not OVERWRITE:
            existing_path = existing_paths[0]
            return record_page_status(
                page,
                "skipped_existing",
                output_path=str(existing_path),
                bytes=existing_path.stat().st_size,
                detected_extension=image_magic_extension(existing_path),
            )

        response, attempts = fetch_with_retries(page["full_image_url"], page_id, auth_stop_event)
        output_path, detected_extension = write_detected_image(page_id, response.content)
        time.sleep(REQUEST_SLEEP_SECONDS)
        return record_page_status(
            page,
            "downloaded",
            output_path=str(output_path),
            bytes=len(response.content),
            detected_extension=detected_extension,
            content_type=response.headers.get("Content-Type", ""),
            final_url=response.url,
            attempts=attempts,
        )
    except AuthenticationFailure as exc:
        with auth_state_lock:
            auth_state["authentication_errors"] += 1
            if auth_state["authentication_errors"] >= AUTH_FAILURE_LIMIT:
                auth_stop_event.set()
            auth_failures = auth_state["authentication_errors"]
        return record_page_status(
            page,
            "authentication_error",
            error=str(exc),
            attempts=MAX_RETRIES,
            authentication_errors_seen=auth_failures,
        )
    except AuthStopRequested as exc:
        return record_page_status(page, "not_processed_auth_stop", error=str(exc))
    except Exception as exc:
        return record_page_status(page, "error", error=repr(exc))


def write_checkpoint(records: list[dict]) -> None:
    if CHECKPOINT_EVERY_RECORDS is None or not records:
        return
    if len(records) % CHECKPOINT_EVERY_RECORDS != 0:
        return
    ordered = sorted(records, key=lambda record: record.get("planned_order", 0))
    CHECKPOINT_JSON.write_text(json.dumps(ordered, indent=2, default=str) + "\n", encoding="utf-8")


def run_serial_downloads(pages: list[dict]) -> list[dict]:
    records = []
    auth_stop_event = Event()
    auth_state = {"authentication_errors": 0}
    auth_state_lock = Lock()

    for index, page in enumerate(pages, start=1):
        record = download_page(page, auth_stop_event, auth_state, auth_state_lock)
        records.append(record)
        print(f"[{index}/{len(pages)}] {page['page_id']}: {record['status']}")
        write_checkpoint(records)
        if auth_stop_event.is_set():
            print(
                f"Stopping downloads after {auth_state['authentication_errors']} authentication-like failure(s). "
                f"Refresh {AUTH_COOKIE_PATH} before continuing."
            )
            for remaining_page in pages[index:]:
                records.append(record_page_status(remaining_page, "not_processed_auth_stop"))
            break

    return records


def run_parallel_downloads(pages: list[dict]) -> list[dict]:
    records = []
    submitted_orders = set()
    completed_orders = set()
    auth_stop_event = Event()
    auth_state = {"authentication_errors": 0}
    auth_state_lock = Lock()
    page_iter = iter(pages)
    pending = set()
    total = len(pages)

    def submit_next(executor: ThreadPoolExecutor) -> bool:
        if auth_stop_event.is_set():
            return False
        try:
            page = next(page_iter)
        except StopIteration:
            return False
        submitted_orders.add(page["planned_order"])
        pending.add(executor.submit(download_page, page, auth_stop_event, auth_state, auth_state_lock))
        return True

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        while len(pending) < max_workers and submit_next(executor):
            pass

        while pending:
            done, pending = wait(pending, return_when=FIRST_COMPLETED)
            for future in done:
                record = future.result()
                records.append(record)
                completed_orders.add(record["planned_order"])
                print(f"[{len(records)}/{total}] {record['page_id']}: {record['status']}")
                write_checkpoint(records)

            while len(pending) < max_workers and submit_next(executor):
                pass

    if auth_stop_event.is_set():
        not_completed = [
            page
            for page in pages
            if page["planned_order"] not in completed_orders
        ]
        already_recorded_orders = {record["planned_order"] for record in records}
        for page in not_completed:
            if page["planned_order"] not in already_recorded_orders:
                records.append(record_page_status(page, "not_processed_auth_stop"))
        print(
            f"Stopped queueing after {auth_state['authentication_errors']} authentication-like failure(s). "
            f"Refresh {AUTH_COOKIE_PATH} before continuing."
        )

    return records


download_records = []
resolved_pages_for_processing = resolved_pages.reset_index(drop=True).copy()
if not resolved_pages_for_processing.empty:
    resolved_pages_for_processing.insert(0, "planned_order", range(1, len(resolved_pages_for_processing) + 1))

pages_to_process = resolved_pages_for_processing.copy()

if DOWNLOAD_IMAGES and TEST_DOWNLOAD_PAGE_LIMIT is not None:
    pages_to_process = pages_to_process.head(int(TEST_DOWNLOAD_PAGE_LIMIT)).copy()
    print(f"TEST_DOWNLOAD_PAGE_LIMIT active: processing {len(pages_to_process):,} of {len(resolved_pages_for_processing):,} planned pages.")

if not DOWNLOAD_IMAGES:
    print("DOWNLOAD_IMAGES is False; writing planned records without network requests.")
    download_records = [
        record_page_status(page, "planned_not_downloaded")
        for page in resolved_pages_for_processing.to_dict("records")
    ]
elif PARALLEL_DOWNLOADS:
    download_records = run_parallel_downloads(pages_to_process.to_dict("records"))
else:
    download_records = run_serial_downloads(pages_to_process.to_dict("records"))

if DOWNLOAD_IMAGES and TEST_DOWNLOAD_PAGE_LIMIT is not None and len(pages_to_process) < len(resolved_pages_for_processing):
    processed_orders = {record["planned_order"] for record in download_records}
    for page in resolved_pages_for_processing.loc[~resolved_pages_for_processing["planned_order"].isin(processed_orders)].to_dict("records"):
        if page["planned_order"] > len(pages_to_process):
            download_records.append(record_page_status(page, "not_processed_test_limit"))

print(f"Records prepared: {len(download_records):,}")


## Save Manifest and Verify Outputs

The manifest records the selected source scan, page ID, METS path, full IIIF URL, output path, status, byte count, and any error. Successful records are verified against local files when downloads are enabled.


In [ ]:
manifest_df = pd.DataFrame(download_records)
if manifest_df.empty:
    manifest_df = pd.DataFrame(columns=list(resolved_pages.columns) + [
        "planned_order",
        "status",
        "output_path",
        "bytes",
        "detected_extension",
        "content_type",
        "final_url",
        "attempts",
        "error",
    ])

if "planned_order" in manifest_df.columns:
    manifest_df = manifest_df.sort_values("planned_order", kind="mergesort").reset_index(drop=True)

manifest_df.to_csv(MANIFEST_CSV, index=False)
manifest_records = json.loads(manifest_df.to_json(orient="records"))
MANIFEST_JSON.write_text(json.dumps(manifest_records, indent=2) + "\n", encoding="utf-8")

error_df = manifest_df.loc[manifest_df["status"].isin(["error", "authentication_error"])].copy()
error_df.to_csv(ERROR_CSV, index=False)
error_records = json.loads(error_df.to_json(orient="records"))
ERROR_JSON.write_text(json.dumps(error_records, indent=2) + "\n", encoding="utf-8")

if DOWNLOAD_IMAGES and not manifest_df.empty:
    output_statuses = {"downloaded", "skipped_existing"}
    output_rows = manifest_df.loc[manifest_df["status"].isin(output_statuses)].copy()
    missing_outputs = output_rows.loc[~output_rows["output_path"].map(lambda value: Path(value).exists() if value else False)]
    assert missing_outputs.empty, missing_outputs[["page_id", "status", "output_path"]].head().to_dict("records")

status_counts = manifest_df["status"].value_counts().to_dict() if "status" in manifest_df else {}
summary = pd.Series(
    {
        "planned_pages": len(resolved_pages),
        "manifest_rows": len(manifest_df),
        "downloaded": int(status_counts.get("downloaded", 0)),
        "skipped_existing": int(status_counts.get("skipped_existing", 0)),
        "not_processed_auth_stop": int(status_counts.get("not_processed_auth_stop", 0)),
        "not_processed_test_limit": int(status_counts.get("not_processed_test_limit", 0)),
        "errors": int(status_counts.get("error", 0)),
        "authentication_errors": int(status_counts.get("authentication_error", 0)),
        "parallel_downloads": bool(PARALLEL_DOWNLOADS),
        "max_workers": int(max_workers),
        "manifest_csv": str(MANIFEST_CSV),
        "manifest_json": str(MANIFEST_JSON),
        "checkpoint_json": str(CHECKPOINT_JSON) if CHECKPOINT_JSON.exists() else None,
        "error_csv": str(ERROR_CSV),
        "output_dir": str(OUTPUT_DIR),
    },
    name="value",
)
display(summary.to_frame())
manifest_df.head(30)


## Conclusion

The notebook creates a reproducible full-page image set from the deduplicated data source. Use the manifest to audit which source scan entries produced each page file and whether a page was downloaded, skipped because it already existed, or not processed due to a test limit or authentication stop.

Parallel mode is available for controlled runs, but the default remains serial. Start with a small `TEST_DOWNLOAD_PAGE_LIMIT` and `MAX_WORKERS = 2` before increasing concurrency.
